## Goodbooks-10k Dataset

First things first, we have to load the dataset, peek into the data and see which columns it has.

In [1]:
import pandas as pd

In [2]:
gb10k_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/refs/heads/master/books.csv"
gb10k = pd.read_csv(gb10k_url)
gb10k.head()

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [3]:
gb10k.columns

Index(['book_id', 'goodreads_book_id', 'best_book_id', 'work_id',
       'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year',
       'original_title', 'title', 'language_code', 'average_rating',
       'ratings_count', 'work_ratings_count', 'work_text_reviews_count',
       'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5',
       'image_url', 'small_image_url'],
      dtype='str')

Our main goal is to collect our training dataset that maps books' content (actual text)
to books' ratings.

The dataset contains a number of columns we are interested in:
- **Columns helping to identify the book**: title, year, authors, isbn, etc. These columns will help us to obtain book's text.
- **Ratings**: we will use average rating as target column when training our model.

Therefore, we have to select only the columns of our interest:

In [4]:
columns_of_interest = [
    "book_id",
    "isbn",
    "isbn13",
    "authors",
    "original_publication_year",
    "original_title",
    "title",
    "language_code",
    "average_rating"
]

gb10k = gb10k[columns_of_interest]
gb10k.head()

,book_id,isbn,isbn13,authors,original_publication_year,original_title,title,language_code,average_rating
0,1,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",eng,4.34
1,2,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44
2,3,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",en-US,3.57
3,4,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,eng,4.25
4,5,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,eng,3.89


In [5]:
gb10k.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   book_id                    10000 non-null  int64  
 1   isbn                       9300 non-null   str    
 2   isbn13                     9415 non-null   float64
 3   authors                    10000 non-null  str    
 4   original_publication_year  9979 non-null   float64
 5   original_title             9415 non-null   str    
 6   title                      10000 non-null  str    
 7   language_code              8916 non-null   str    
 8   average_rating             10000 non-null  float64
dtypes: float64(3), int64(1), str(5)
memory usage: 703.3 KB


As you can see, pandas misreads data types for certain columns: e.g. it read isbn13 as float64 and original_publication_year as float64. Thus, we have to specify correct data types. Additionally, we can rename columns of interest to a more suitable set of names. It will become useful if we'll decide to add more rating datasets in the future, and they will have different columns.

In [6]:
gb10k_column_mapper = {
    "book_id": "id",
    "original_publication_year": "year",
    "language_code": "language",
    "average_rating": "rating"
}

gb10k = gb10k.rename(columns=gb10k_column_mapper)
gb10k["year"] = gb10k["year"].astype("Int64")
gb10k["isbn13"] = gb10k["isbn13"].astype("Int64").astype("str").str.pad(width=13, side="left", fillchar="0")
gb10k.head()

,id,isbn,isbn13,authors,year,original_title,title,language,rating
0,1,439023483,9780439023480,Suzanne Collins,2008,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",eng,4.34
1,2,439554934,9780439554930,"J.K. Rowling, Mary GrandPré",1997,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44
2,3,316015849,9780316015840,Stephenie Meyer,2005,Twilight,"Twilight (Twilight, #1)",en-US,3.57
3,4,61120081,9780061120080,Harper Lee,1960,To Kill a Mockingbird,To Kill a Mockingbird,eng,4.25
4,5,743273567,9780743273560,F. Scott Fitzgerald,1925,The Great Gatsby,The Great Gatsby,eng,3.89


In [7]:
gb10k.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              10000 non-null  int64  
 1   isbn            9300 non-null   str    
 2   isbn13          9415 non-null   str    
 3   authors         10000 non-null  str    
 4   year            9979 non-null   Int64  
 5   original_title  9415 non-null   str    
 6   title           10000 non-null  str    
 7   language        8916 non-null   str    
 8   rating          10000 non-null  float64
dtypes: Int64(1), float64(1), int64(1), str(6)
memory usage: 713.0 KB


Now, when we got the dtypes that we wanted and the columns that we wanted, it's time to explore the data more in-depth.
The questions that we want to address are the following:
1. Are `id`, `title` (`original_title`), `isbn` and `isbn13` unique in the dataset? In other words, does each book occur only once? What can we count as "book identity"?
2. Does `year` column contain valid years? What's the range of years that we have?
3. What is the distribution of books across the years?
4. What is the difference between `original_title` and `title`? When do these columns differ?
5. `authors` column does not contain any nulls. Is it well-formatted? Are there any empty or malformed strings?
6. In general, are string columns well-formatted?
7. What are the languages that we have in the `language` column? Can two language codes map to the same language? What kind of language code format does it use?
8. What's the distribution of `rating`? Is the dataset balanced?

Another thing would be **transforming the dataset** in order to fix the issues, and that'll be done as a separate part.

In [22]:
for col, typ in gb10k.dtypes.items():
    if typ != "str":
        continue
    empty_count = (gb10k[col].str.strip() == "").sum()
    print(f"{col}: {empty_count} empty")

isbn: 0 empty
isbn13: 0 empty
authors: 0 empty
original_title: 5 empty
title: 0 empty
language: 0 empty


In [58]:
titles = gb10k[["title", "original_title"]]
titles[titles["title"] != titles["original_title"]]

,title,original_title
0,"The Hunger Games (The Hunger Games, #1)",The Hunger Games
1,Harry Potter and the Sorcerer's Stone (Harry P...,Harry Potter and the Philosopher's Stone
2,"Twilight (Twilight, #1)",Twilight
6,The Hobbit,The Hobbit or There and Back Again
8,"Angels & Demons (Robert Langdon, #1)",Angels & Demons
...,...,...
9988,"Out of the Dark (The Grey Wolves, #4)",Out of the Dark
9991,"The Red Knight (The Traitor Son Cycle, #1)",The Red Knight
9992,The Time Traveller's Guide to Medieval England...,The Time-Traveller's Guide to Medieval England...
9995,"Bayou Moon (The Edge, #2)",Bayou Moon


In [46]:
import re

isbn10_pattern = re.compile(r"^\d{9}(\d|X)$")
isbn13_pattern = re.compile(r"^\d{13}$")

def check_isbn10(value):
    return bool(isbn10_pattern.match(value))

def check_isbn13(value):
    return bool(isbn13_pattern.match(value))

In [49]:
isbn = gb10k["isbn"]
isbn = isbn[isbn.notna()]
isbn[~isbn.apply(check_isbn10)]

0       439023483
1       439554934
2       316015849
3        61120081
4       743273567
          ...    
9992    224079948
9993    810983559
9995    441019455
9998     61711527
9999    375700455
Name: isbn, Length: 6601, dtype: str

In [50]:
isbn13 = gb10k["isbn13"]
isbn13 = isbn13[isbn13.notna()]
isbn13[~isbn13.apply(check_isbn13)]

Series([], Name: isbn13, dtype: str)

In [57]:
def check_uniqueness(col_name):
    unique = gb10k[col_name].unique().size
    notnull = gb10k[col_name].notna().size
    return (unique / notnull) * 100.0

for col in ["id", "title", "original_title", "isbn", "isbn13"]:
    uniqueness = check_uniqueness(col)
    print(f"{col} uniqueness: {uniqueness}%")

id uniqueness: 100.0%
title uniqueness: 99.64%
original_title uniqueness: 92.75%
isbn uniqueness: 93.01%
isbn13 uniqueness: 91.53999999999999%
